# Stage-1 Llama runner

This notebook runs the Stage-1 deceptive-reasoning capture path from the GitHub repository. It does not run a GPT-2 smoke path; the local GPT-2 smoke already validates basic plumbing. T4 can be used for a constrained one-layer attempt with batch size 1, while A100 or another high-memory GPU is the realistic target for full Llama sweeps.

Harmful compliance remains quarantined and is not part of this Stage-1 run.

## 1. Parameters

Set these before running the notebook. Keep `SPLIT = 'train'` for vector fitting. Do not capture `test` unless the analysis has been explicitly frozen.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ashioyajotham/safety_governor.git'
BRANCH = 'main'
REPO_DIR = Path('/content/safety_governor')
CONFIG_PATH = 'configs/llama3_8b.yaml'
SPLIT = 'train'
LAYER = 0
BATCH_SIZE = 1
DEVICE = 'cuda'
VECTOR_METHOD = 'difference_in_means'
BOOTSTRAP_SAMPLES = 100
ARTIFACT_ROOT = Path('/content/drive/MyDrive/safety_governor_stage1')
RUN_CAPTURE = True
RUN_VECTOR = True

# Optional later expansion after the first layer succeeds.
RUN_LAYER_SWEEP = False
SWEEP_LAYERS = [0, 4, 8, 12, 16, 20, 24, 28]


## 2. Clone or pull repository

In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command, cwd=None, env=None):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True)

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)
else:
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
run(['git', 'rev-parse', 'HEAD'])
run(['git', 'status', '--short'])


## 3. Install runtime and authenticate Hugging Face

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('Hugging Face token loaded from Colab secrets.')
else:
    print('No HF_TOKEN Colab secret found. Llama access may fail if the model is gated.')


## 4. Runtime and corpus preflight

In [ ]:
import collections
import json
import platform

run(['nvidia-smi'])
run([sys.executable, '-m', 'scripts.validate_dataset', 'datasets/frozen/english_contrastive.jsonl'])
run([sys.executable, '-m', 'scripts.verify_environment', CONFIG_PATH])

rows = [json.loads(line) for line in Path('datasets/frozen/english_contrastive.jsonl').read_text(encoding='utf-8').splitlines() if line]
print('python', sys.version)
print('platform', platform.platform())
print('rows', len(rows), 'pairs', len({row['pair_id'] for row in rows}))
print('behaviors', collections.Counter(row['behavior'] for row in rows))
print('archetypes', collections.Counter(row.get('archetype') for row in rows))
print('splits', collections.Counter(row['split'] for row in rows))
print('polarities', collections.Counter(row['polarity'] for row in rows))


## 5. One-layer Llama capture

In [ ]:
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = ARTIFACT_ROOT / datetime.now(timezone.utc).strftime('stage1_llama_%Y%m%dT%H%M%SZ')
run_dir.mkdir(parents=True, exist_ok=False)
print('artifact root:', run_dir)

if RUN_CAPTURE:
    run([
        sys.executable, '-m', 'scripts.capture_activations', CONFIG_PATH,
        '--layer', str(LAYER),
        '--split', SPLIT,
        '--device', DEVICE,
        '--batch-size', str(BATCH_SIZE),
        '--artifacts', str(run_dir),
    ])
else:
    print('RUN_CAPTURE is false; skipping capture.')

manifests = sorted(run_dir.glob('capture-*/manifest.json'))
if not manifests:
    raise FileNotFoundError('No capture manifest found under artifact root.')
manifest_path = manifests[-1]
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('manifest:', manifest_path)
print(json.dumps({
    'run_id': manifest['run_id'],
    'pairs': manifest['metrics']['pairs'],
    'capture_layer': manifest['config']['capture_layer'],
    'capture_split': manifest['config']['capture_split'],
    'capture_site': manifest['config']['capture_site'],
    'capture_batch_size': manifest['config'].get('capture_batch_size'),
    'dataset_sha256': manifest['config']['dataset_sha256'],
    'model_revision': manifest['config']['model']['revision'],
}, indent=2))


## 6. Vector extraction

In [ ]:
if RUN_VECTOR:
    safe_path = manifest['artifacts']['safe_activations']
    unsafe_path = manifest['artifacts']['unsafe_activations']
    vector_path = str(Path(manifest_path).parent / f'{VECTOR_METHOD}_vector.npy')
    run([
        sys.executable, '-m', 'scripts.extract_vector',
        '--safe', safe_path,
        '--unsafe', unsafe_path,
        '--method', VECTOR_METHOD,
        '--output', vector_path,
        '--bootstrap-samples', str(BOOTSTRAP_SAMPLES),
    ])
    print('vector:', vector_path)
else:
    print('RUN_VECTOR is false; skipping vector extraction.')


## 7. Optional layer sweep

Run this only after the one-layer capture succeeds. Keep `SPLIT = 'train'`; validation/test are downstream evaluation splits.

In [ ]:
if RUN_LAYER_SWEEP:
    for layer in SWEEP_LAYERS:
        layer_dir = run_dir / f'layer_{layer:02d}'
        layer_dir.mkdir(parents=True, exist_ok=False)
        run([
            sys.executable, '-m', 'scripts.capture_activations', CONFIG_PATH,
            '--layer', str(layer),
            '--split', SPLIT,
            '--device', DEVICE,
            '--batch-size', str(BATCH_SIZE),
            '--artifacts', str(layer_dir),
        ])
        layer_manifest_path = sorted(layer_dir.glob('capture-*/manifest.json'))[-1]
        layer_manifest = json.loads(layer_manifest_path.read_text(encoding='utf-8'))
        run([
            sys.executable, '-m', 'scripts.extract_vector',
            '--safe', layer_manifest['artifacts']['safe_activations'],
            '--unsafe', layer_manifest['artifacts']['unsafe_activations'],
            '--method', VECTOR_METHOD,
            '--output', str(Path(layer_manifest_path).parent / f'{VECTOR_METHOD}_vector.npy'),
            '--bootstrap-samples', str(BOOTSTRAP_SAMPLES),
        ])
else:
    print('RUN_LAYER_SWEEP is false; one-layer run only.')


## 8. Artifact summary

In [ ]:
for path in sorted(run_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(run_dir), path.stat().st_size)
